# So Sánh 4 Model Kiểm Duyệt Ảnh — CNN · Transformer · Multimodal (CLIP)

**Task:** Phân loại ảnh thành 3 class cho hệ thống kiểm duyệt nội dung:
- `SAFE` (0) — ảnh bình thường, an toàn
- `NSFW` (1) — nội dung 18+, khiêu dâm
- `VIOLENCE` (2) — bạo lực, máu me

---

**4 Model so sánh:**

| # | Model | Kiểu | Params | Đặc điểm |
|---|-------|------|--------|-----------|
| 1 | EfficientNet-B0 | CNN | ~5M | Nhẹ, nhanh, production-friendly |
| 2 | ResNet-50 | CNN | ~25M | Baseline kinh điển |
| 3 | ViT-base-patch16 | Image Transformer | ~86M | Attention toàn cục |
| 4 | CLIP-ViT-B/32 | **Multimodal** | ~150M | Hiểu cả ảnh + ngữ nghĩa text |

CLIP được chạy ở 2 chế độ: **zero-shot** (không train) và **fine-tuned** — để thấy rõ giá trị của fine-tuning.

---

**Pipeline:**
```
1. Load raw ảnh
2. Kiểm tra chất lượng (corrupt, kích thước, duplicate)
3. Stratified split 80/10/10 → TRƯỚC khi augment
4. Augment chỉ trên train set (VIOLENCE upsample lên 10k)
5. Tính mean/std thực tế từ train
6. CLIP zero-shot baseline
7. LR Finder cho từng model → LR tối ưu khách quan
8. Train 4 models
9. Đánh giá toàn diện: ROC · PR · Calibration · Error Analysis
10. Grad-CAM · ViT Attention · CLIP Attention · t-SNE · Radar chart
```

**Yêu cầu Kaggle:**
- Accelerator: **GPU T4 × 2** hoặc P100
- Internet: **ON**
- Add Dataset: tìm **"Real Life Violence Situations Dataset"** > Add

**Ước tính:** ~6–8 giờ tổng

In [ ]:
!pip install timm datasets scikit-learn torch-lr-finder grad-cam transformers imagehash -q

In [ ]:
import os, json, random, time, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
from datasets import load_dataset
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score
)
from sklearn.manifold import TSNE
from pathlib import Path
from PIL import Image
from collections import Counter
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import imagehash

# Multimodal
from transformers import CLIPModel, CLIPProcessor

# Explainability
from pytorch_grad_cam import GradCAM, GradCAMPlusPlus
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# LR Finder
from torch_lr_finder import LRFinder

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

OUTPUT_DIR = Path('/kaggle/working')
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
# ════════════════════════════════════════════════════════════
# CẤU HÌNH CHUNG
# ════════════════════════════════════════════════════════════
SEED         = 42
IMG_SIZE     = 224
BATCH_SIZE   = 64
EPOCHS       = 10
NUM_WORKERS  = 2
PATIENCE     = 3

CLASS_NAMES    = ['SAFE', 'NSFW', 'VIOLENCE']
NUM_CLASSES    = 3
LABEL_SAFE     = 0
LABEL_NSFW     = 1
LABEL_VIOLENCE = 2
MAX_PER_CLASS  = 10_000

# Normalization constants
MEAN_IMAGENET = [0.485, 0.456, 0.406]
STD_IMAGENET  = [0.229, 0.224, 0.225]
MEAN_CLIP     = [0.48145466, 0.4578275, 0.40821073]   # CLIP dùng riêng
STD_CLIP      = [0.26862954, 0.26130258, 0.27577711]

# 3 CNN/ViT model + CLIP được xử lý riêng
CNN_MODELS = {
    'efficientnet_b0'     : 'EfficientNet-B0',
    'resnet50'            : 'ResNet-50',
    'vit_base_patch16_224': 'ViT-Base',
}
CLIP_MODEL_NAME = 'openai/clip-vit-base-patch32'

# LR sẽ được xác định bởi LR Finder — đây chỉ là giá trị khởi đầu tìm kiếm
LR_SEARCH_START = 1e-7
LR_SEARCH_END   = 1.0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('Config loaded.')
print(f'  Classes       : {CLASS_NAMES}')
print(f'  Max per class : {MAX_PER_CLASS:,}')
print(f'  CNN models    : {list(CNN_MODELS.keys())}')
print(f'  CLIP model    : {CLIP_MODEL_NAME}')

In [ ]:
# ════════════════════════════════════════════════════════════
# BƯỚC 1: LOAD RAW IMAGES
#   SAFE     : Kaggle folder (auto-scan)
#   NSFW     : wallstoneai/civitai-top-nsfw-images-with-metadata
#              + DarkyMan/nsfw-image-classification (bo sung)
#   VIOLENCE : abdulmananraja (Kaggle)
# ════════════════════════════════════════════════════════════
import os, re, io
from pathlib import Path

raw_by_class = {LABEL_SAFE: [], LABEL_NSFW: [], LABEL_VIOLENCE: []}
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
MAX_LOAD_PER_CLASS = 5_000
print(f'MAX_LOAD_PER_CLASS = {MAX_LOAD_PER_CLASS:,}')

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    print(f'HF_TOKEN: OK ({HF_TOKEN[:8]}...)')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')
    print(f'HF_TOKEN: {"OK" if HF_TOKEN else "not set"}')

# ════════════════════════════════════════════════════════════
# A: SAFE — scan Kaggle folders
# ════════════════════════════════════════════════════════════
print('\n[A] SAFE — Kaggle folder scan...')
SAFE_TOKENS = {'safe', 'sfw', 'neutral', 'normal', 'clean'}
NSFW_TOKENS = {'nsfw', 'explicit', 'porn', 'nude', 'hentai', 'adult', 'erotic', 'r18', 'sexy'}

safe_dir = None
for root, dirs, files in os.walk('/kaggle/input'):
    n_imgs = sum(1 for f in files if Path(f).suffix.lower() in IMG_EXTS)
    if n_imgs < 5:
        continue
    tokens = set(re.split(r'[^a-z0-9]+', Path(root).name.lower()))
    if tokens & NSFW_TOKENS:
        continue
    if tokens & SAFE_TOKENS and 'not' not in tokens:
        safe_dir = root
        print(f'  SAFE dir: {root}  ({n_imgs} imgs)')
        break

if safe_dir:
    files = [f for f in Path(safe_dir).rglob('*') if f.suffix.lower() in IMG_EXTS]
    random.shuffle(files)
    for fp in tqdm(files[:MAX_LOAD_PER_CLASS], desc='  SAFE'):
        try:
            raw_by_class[LABEL_SAFE].append(Image.open(fp).convert('RGB'))
        except Exception:
            continue
    print(f'  SAFE: {len(raw_by_class[LABEL_SAFE]):,}')
else:
    print('  [!] SAFE dir NOT FOUND. /kaggle/input:')
    for p in sorted(Path('/kaggle/input').iterdir()):
        if p.is_dir():
            print(f'    {p.name}/ -> {[x.name for x in p.iterdir() if x.is_dir()][:6]}')

# ════════════════════════════════════════════════════════════
# B: NSFW — nhieu nguon, chay theo thu tu den khi du 5000
# ════════════════════════════════════════════════════════════
print('\n[B] NSFW — multi-source streaming...')

NSFW_SOURCES = [
    'wallstoneai/civitai-top-nsfw-images-with-metadata',  # all NSFW, public
    'DarkyMan/nsfw-image-classification',                 # ~600 anh bo sung
    'x1101/nsfw-full',                                      # bo sung them
]

def _load_nsfw_from_hf(dataset_id, need):
    """Stream anh NSFW tu HF dataset. Tra ve so anh da them."""
    from datasets import load_dataset
    kw = dict(path=dataset_id, split='train', streaming=True)
    if HF_TOKEN:
        kw['token'] = HF_TOKEN
    ds = load_dataset(**kw)
    cols = list(ds.features.keys())
    print(f'    features: {cols}')

    # Tim cot anh
    img_col = next(
        (c for c in cols if c.lower() in ('image', 'img', 'photo', 'thumbnail')),
        next((c for c in cols if any(k in c.lower() for k in ('image','img','photo'))), None)
    )
    print(f'    image col: {img_col}')

    added = n_iter = 0
    for sample in ds:
        if added >= need:
            break
        n_iter += 1
        try:
            raw = sample.get(img_col) if img_col else None
            if raw is None:
                for v in sample.values():
                    if isinstance(v, (Image.Image, dict, bytes)):
                        raw = v; break
            if isinstance(raw, Image.Image):
                img = raw.convert('RGB')
            elif isinstance(raw, dict) and raw.get('bytes'):
                img = Image.open(io.BytesIO(raw['bytes'])).convert('RGB')
            elif isinstance(raw, bytes):
                img = Image.open(io.BytesIO(raw)).convert('RGB')
            else:
                continue
            raw_by_class[LABEL_NSFW].append(img)
            added += 1
        except Exception:
            continue
        if n_iter % 500 == 0:
            print(f'    iter={n_iter:,}  +{added:,}')
    return added

for src in NSFW_SOURCES:
    cur = len(raw_by_class[LABEL_NSFW])
    if cur >= MAX_LOAD_PER_CLASS:
        break
    need = MAX_LOAD_PER_CLASS - cur
    print(f'  [{src}]  can them {need:,} anh...')
    try:
        added = _load_nsfw_from_hf(src, need)
        print(f'  -> +{added:,}  tong NSFW={len(raw_by_class[LABEL_NSFW]):,}')
    except Exception as e:
        import traceback
        print(f'  -> fail: {type(e).__name__}: {str(e)[:200]}')
        traceback.print_exc()

print(f'  NSFW final: {len(raw_by_class[LABEL_NSFW]):,}')
if len(raw_by_class[LABEL_NSFW]) == 0:
    raise RuntimeError('Khong load duoc NSFW!')

# ════════════════════════════════════════════════════════════
# C: VIOLENCE — abdulmananraja/real-life-violence-situations
# ════════════════════════════════════════════════════════════
print('\n[C] VIOLENCE — abdulmananraja...')

VIOLENCE_DIRS = [
    '/kaggle/input/datasets/abdulmananraja/real-life-violence-situations/new_violence/violence',
    '/kaggle/input/datasets/abdulmananraja/real-life-violence-situations/new_violence/Violence',
    '/kaggle/input/real-life-violence-situations/new_violence/violence',
    '/kaggle/input/real-life-violence-situations/Violence',
    '/kaggle/input/real-life-violence-situations/violence',
]
violence_dir = next((d for d in VIOLENCE_DIRS if os.path.isdir(d)), None)

if not violence_dir:
    for root, dirs, flist in os.walk('/kaggle/input'):
        if any(Path(f).suffix.lower() in IMG_EXTS for f in flist):
            t = set(re.split(r'[^a-z0-9]+', Path(root).name.lower()))
            if t & {'violence', 'violent', 'fight', 'fighting'}:
                violence_dir = root; break

if violence_dir:
    files = [f for f in Path(violence_dir).rglob('*') if f.suffix.lower() in IMG_EXTS]
    random.shuffle(files)
    print(f'  {len(files):,} imgs in {violence_dir}')
    for fp in tqdm(files[:MAX_LOAD_PER_CLASS], desc='  Violence'):
        try:
            raw_by_class[LABEL_VIOLENCE].append(Image.open(fp).convert('RGB'))
        except Exception:
            continue
    print(f'  VIOLENCE: {len(raw_by_class[LABEL_VIOLENCE]):,}')
else:
    print('  [!] NOT FOUND')

# ════════════════════════════════════════════════════════════
# Tom tat
# ════════════════════════════════════════════════════════════
n_s = len(raw_by_class[LABEL_SAFE])
n_n = len(raw_by_class[LABEL_NSFW])
n_v = len(raw_by_class[LABEL_VIOLENCE])
print(f'\nRaw: SAFE={n_s:,} | NSFW={n_n:,} | VIOLENCE={n_v:,}')
_min = min(n_s, n_n, max(n_v, 1))
_max = max(n_s, n_n, n_v)
ratio = _max / _min if _min > 0 else float('inf')
print(f'{"[OK]" if ratio <= 3 else "[!] Mat can bang"} ratio={ratio:.1f}x')


In [ ]:
# ════════════════════════════════════════════════════════════
# BƯỚC 2: KIỂM TRA CHẤT LƯỢNG DỮ LIỆU
# ════════════════════════════════════════════════════════════
print('=== Data Quality Check ===\n')

MIN_SIZE = 64   # ảnh nhỏ hơn 64px coi là quá nhỏ

qa_report = {}
clean_by_class = {LABEL_SAFE: [], LABEL_NSFW: [], LABEL_VIOLENCE: []}

for lbl, imgs in raw_by_class.items():
    name = CLASS_NAMES[lbl]
    too_small = corrupt = 0
    widths, heights = [], []

    for img in tqdm(imgs, desc=f'  QA [{name}]', leave=False):
        try:
            w, h = img.size
            if w < MIN_SIZE or h < MIN_SIZE:
                too_small += 1
                continue
            widths.append(w)
            heights.append(h)
            clean_by_class[lbl].append(img)
        except Exception:
            corrupt += 1

    qa_report[name] = {
        'total': len(imgs), 'kept': len(clean_by_class[lbl]),
        'too_small': too_small, 'corrupt': corrupt,
        'avg_w': int(np.mean(widths)) if widths else 0,
        'avg_h': int(np.mean(heights)) if heights else 0,
    }
    r = qa_report[name]
    print(f'[{name}] total={r["total"]:,} | kept={r["kept"]:,} | '
          f'too_small={r["too_small"]} | corrupt={r["corrupt"]} | '
          f'avg_size={r["avg_w"]}×{r["avg_h"]}')

# ── Perceptual hash dedup (chỉ kiểm tra cross-class) ──────
print('\nKiểm tra cross-class near-duplicate (pHash)...')
hash_to_class = {}
cross_dups = 0
for lbl in [LABEL_SAFE, LABEL_NSFW, LABEL_VIOLENCE]:
    for img in tqdm(clean_by_class[lbl], desc=f'  hashing {CLASS_NAMES[lbl]}', leave=False):
        h = str(imagehash.phash(img.resize((32, 32))))
        if h in hash_to_class and hash_to_class[h] != lbl:
            cross_dups += 1
        else:
            hash_to_class[h] = lbl
print(f'  Cross-class near-duplicates phát hiện: {cross_dups}')

# ── Phân phối kích thước ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['#2196F3', '#FF5722', '#4CAF50']
for lbl in range(NUM_CLASSES):
    ws = [img.size[0] for img in clean_by_class[lbl][:500]]
    hs = [img.size[1] for img in clean_by_class[lbl][:500]]
    axes[0].hist(ws, bins=30, alpha=0.5, color=colors[lbl], label=CLASS_NAMES[lbl])
    axes[1].hist(hs, bins=30, alpha=0.5, color=colors[lbl], label=CLASS_NAMES[lbl])
axes[0].set_title('Phân phối Width'); axes[0].set_xlabel('px'); axes[0].legend()
axes[1].set_title('Phân phối Height'); axes[1].set_xlabel('px'); axes[1].legend()
plt.suptitle('Kích thước ảnh theo class (sample 500/class)', fontweight='bold')
plt.tight_layout(); plt.savefig(OUTPUT_DIR/'size_dist.png', bbox_inches='tight'); plt.show()
print('Saved: size_dist.png')

In [ ]:
# ════════════════════════════════════════════════════════════
# BƯỚC 3: STRATIFIED SPLIT TRƯỚC KHI AUGMENT
# Quan trọng: val/test chỉ dùng ảnh gốc, không có ảnh augmented
# ════════════════════════════════════════════════════════════
def stratified_split(imgs_by_class, train_ratio=0.8, val_ratio=0.1, seed=42):
    rng = random.Random(seed)
    train, val, test = [], [], []
    for lbl, imgs in imgs_by_class.items():
        shuffled = imgs[:]
        rng.shuffle(shuffled)
        n = len(shuffled)
        if n == 0:
            continue
        n_train = int(n * train_ratio)
        n_val   = int(n * val_ratio)
        train += [(img, lbl) for img in shuffled[:n_train]]
        val   += [(img, lbl) for img in shuffled[n_train:n_train + n_val]]
        test  += [(img, lbl) for img in shuffled[n_train + n_val:]]
    rng.shuffle(train); rng.shuffle(val); rng.shuffle(test)
    return train, val, test

train_raw, val_data, test_data = stratified_split(clean_by_class)

# Cảnh báo nếu thiếu class
present = {lbl for _, lbl in train_raw}
missing = [CLASS_NAMES[i] for i in range(NUM_CLASSES) if i not in present]
if missing:
    print(f'[!] Class bị thiếu trong train: {missing}')
    print('    Kiểm tra lại cell Load Raw Images — dataset có thể chưa load đủ')

print('Split TRƯỚC augment:')
for name, data in [('Train (raw)', train_raw), ('Val', val_data), ('Test', test_data)]:
    dist = Counter(lbl for _, lbl in data)
    print(f'  {name:15s}: {len(data):,} | '
          + ' | '.join(f'{CLASS_NAMES[i]}={dist[i]:,}' for i in range(NUM_CLASSES)))

# ── Augment chỉ áp dụng cho train set → upsample VIOLENCE ──────
aug_for_balance = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.25, hue=0.05),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.70, 1.0)),
    transforms.RandomGrayscale(p=0.05),
])

train_dist   = Counter(lbl for _, lbl in train_raw)
train_data   = list(train_raw)
violence_raw = [img for img, lbl in train_raw if lbl == LABEL_VIOLENCE]

if not violence_raw:
    print('\n[!] Không có ảnh VIOLENCE trong train — bỏ qua bước augment VIOLENCE')
    print('    Thêm violence dataset trước khi train để có kết quả đầy đủ')
else:
    target_per_class = max(train_dist.get(LABEL_SAFE, 0),
                           train_dist.get(LABEL_NSFW, 0),
                           len(violence_raw))
    needed = target_per_class - len(violence_raw)
    if needed > 0:
        print(f'\nAugment VIOLENCE train: {len(violence_raw):,} → {target_per_class:,} (+{needed:,})')
        random.seed(SEED)
        for i in range(needed):
            src = violence_raw[i % len(violence_raw)]
            train_data.append((aug_for_balance(src), LABEL_VIOLENCE))
        random.shuffle(train_data)
    else:
        print('\nVIOLENCE đã đủ, không cần augment')

print('\nSau augment:')
aug_dist = Counter(lbl for _, lbl in train_data)
for i, name in enumerate(CLASS_NAMES):
    orig = train_dist.get(i, 0)
    aug  = aug_dist.get(i, 0) - orig
    print(f'  {name}: {aug_dist.get(i,0):,}  (gốc={orig:,} + augmented={aug:,})')
print(f'  Tổng train: {len(train_data):,} | Val: {len(val_data):,} | Test: {len(test_data):,}')


In [ ]:
# ════════════════════════════════════════════════════════════
# BƯỚC 4: DATASET CLASS + DATALOADERS
# ════════════════════════════════════════════════════════════

# ── 4a: Tính mean/std thực tế từ train set ───────────────
print('Tính mean/std từ train set (sample 2000 ảnh)...')
sample_imgs = random.sample(train_data, min(2000, len(train_data)))
tensor_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])
stack = torch.stack([tensor_transform(img) for img, _ in sample_imgs])  # (N, C, H, W)
MEAN_ACTUAL = stack.mean(dim=[0,2,3]).tolist()
STD_ACTUAL  = stack.std(dim=[0,2,3]).tolist()
print(f'  Mean thực tế  : {[round(x,4) for x in MEAN_ACTUAL]}')
print(f'  Std thực tế   : {[round(x,4) for x in STD_ACTUAL]}')
print(f'  Mean ImageNet : {MEAN_IMAGENET}')
diff = max(abs(MEAN_ACTUAL[i] - MEAN_IMAGENET[i]) for i in range(3))
print(f'  → Chênh lệch max: {diff:.4f} — {"dùng ImageNet default OK" if diff < 0.05 else "nên dùng mean/std thực tế"}')
MEAN_USE = MEAN_ACTUAL if diff >= 0.05 else MEAN_IMAGENET
STD_USE  = STD_ACTUAL  if diff >= 0.05 else STD_IMAGENET
print(f'  Sử dụng: {MEAN_USE}')

# ── 4b: Transforms ────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN_USE, STD_USE),
])
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN_USE, STD_USE),
])

# ── 4c: Dataset Class cho CNN/ViT ─────────────────────────
class ImageDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data = data
        self.transform = transform
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        img, lbl = self.data[idx]
        if not isinstance(img, Image.Image): img = Image.fromarray(img)
        img = img.convert('RGB')
        if self.transform: img = self.transform(img)
        return img, lbl

train_ds = ImageDataset(train_data, train_transform)
val_ds   = ImageDataset(val_data,   val_transform)
test_ds  = ImageDataset(test_data,  val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

# ── 4d: CLIP-specific Dataset ─────────────────────────────
print('\nLoading CLIP processor...')
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)

class CLIPDataset(Dataset):
    def __init__(self, data, processor):
        self.data = data
        self.processor = processor
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        img, lbl = self.data[idx]
        if not isinstance(img, Image.Image): img = Image.fromarray(img)
        inputs = self.processor(images=img.convert('RGB'), return_tensors='pt')
        return inputs['pixel_values'].squeeze(0), lbl

clip_train_ds = CLIPDataset(train_data, clip_processor)
clip_val_ds   = CLIPDataset(val_data,   clip_processor)
clip_test_ds  = CLIPDataset(test_data,  clip_processor)

clip_train_loader = DataLoader(clip_train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=True)
clip_val_loader   = DataLoader(clip_val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=True)
clip_test_loader  = DataLoader(clip_test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=True)

print(f'\nDataLoaders sẵn sàng:')
print(f'  Train: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}')
print(f'  CLIP train: {len(clip_train_ds):,}')

In [ ]:
# ════════════════════════════════════════════════════════════
# BƯỚC 5: CLIP ZERO-SHOT BASELINE
# Không train gì cả — đánh giá CLIP với text prompts để biết
# "điểm xuất phát" của multimodal model trước fine-tuning
# ════════════════════════════════════════════════════════════
print('Loading CLIP for zero-shot evaluation...')
clip_base_zs = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(DEVICE)
clip_base_zs.eval()

ZS_PROMPTS = [
    "a safe and normal everyday photo",
    "a sexually explicit or adult NSFW image",
    "a violent image with fighting blood or weapons",
]

text_inputs_zs = clip_processor(
    text=ZS_PROMPTS, return_tensors='pt', padding=True, truncation=True
).to(DEVICE)
with torch.no_grad():
    text_features_zs = clip_base_zs.get_text_features(**text_inputs_zs)
    text_features_zs = F.normalize(text_features_zs, dim=-1)  # (3, 512)

all_preds_zs, all_labels_zs = [], []
print('Running zero-shot on test set...')
for imgs, labels in tqdm(clip_test_loader, desc='Zero-shot'):
    imgs = imgs.to(DEVICE)
    with torch.no_grad():
        img_feats = clip_base_zs.get_image_features(pixel_values=imgs)
        img_feats  = F.normalize(img_feats, dim=-1)
        sims       = img_feats @ text_features_zs.T
        preds      = sims.argmax(dim=1)
    all_preds_zs.extend(preds.cpu().numpy())
    all_labels_zs.extend(labels.numpy())

zs_acc = accuracy_score(all_labels_zs, all_preds_zs) * 100
zs_f1  = f1_score(all_labels_zs, all_preds_zs, average='macro') * 100
print(f'\n[CLIP Zero-Shot] Acc={zs_acc:.2f}%  Macro F1={zs_f1:.2f}%')
print(classification_report(all_labels_zs, all_preds_zs, target_names=CLASS_NAMES))

# Khởi tạo all_results — sẽ được bổ sung sau khi train
all_results = {}
all_results['clip_zeroshot'] = {
    'model_name' : 'CLIP (zero-shot)',
    'test_acc'   : zs_acc,
    'test_f1'    : zs_f1,
    'test_preds' : all_preds_zs,
    'test_labels': all_labels_zs,
}

del clip_base_zs
torch.cuda.empty_cache()
print('Zero-shot done. all_results khởi tạo.')

In [ ]:
# ════════════════════════════════════════════════════════════
# BƯỚC 6: LR FINDER — Xác định LR tối ưu cho từng model
# Mỗi model được so sánh ở LR tốt nhất của nó → so sánh công bằng
# ════════════════════════════════════════════════════════════
OPTIMAL_LRS = {}

# Dùng 1 000 samples để tìm LR nhanh
lr_subset = random.sample(train_data, min(1000, len(train_data)))
lr_ds_tmp  = ImageDataset(lr_subset, train_transform)
lr_loader_tmp = DataLoader(lr_ds_tmp, batch_size=32, shuffle=True, num_workers=0)

lc_sub = Counter(lbl for _, lbl in lr_subset)
cw_sub = torch.tensor([len(lr_subset) / (NUM_CLASSES * max(lc_sub[i], 1))
                        for i in range(NUM_CLASSES)], dtype=torch.float).to(DEVICE)
crit_sub = nn.CrossEntropyLoss(weight=cw_sub)

fig, axes_lr = plt.subplots(1, 4, figsize=(20, 4))
lr_colors = ['#2196F3', '#4CAF50', '#FF5722', '#9C27B0']

for ax, color, model_key in zip(axes_lr[:3], lr_colors[:3], CNN_MODELS.keys()):
    print(f'LR Finder: {CNN_MODELS[model_key]}...')
    m_tmp = timm.create_model(model_key, pretrained=True, num_classes=NUM_CLASSES).to(DEVICE)
    opt_tmp = torch.optim.AdamW(m_tmp.parameters(), lr=LR_SEARCH_START, weight_decay=1e-4)
    finder  = LRFinder(m_tmp, opt_tmp, crit_sub, device=DEVICE)
    finder.range_test(lr_loader_tmp, start_lr=LR_SEARCH_START, end_lr=LR_SEARCH_END,
                      num_iter=100, step_mode='exp')
    lrs_h   = finder.history['lr']
    loss_h  = finder.history['loss']
    # Lấy LR ở điểm loss giảm nhanh nhất (gradient nhỏ nhất), /10 để lùi an toàn
    smooth  = np.convolve(loss_h, np.ones(5)/5, mode='valid')
    best_i  = int(np.gradient(smooth).argmin()) + 2
    best_i  = min(best_i, len(lrs_h) - 1)
    opt_lr  = lrs_h[best_i] / 10
    OPTIMAL_LRS[model_key] = opt_lr
    ax.plot(lrs_h, loss_h, color=color, lw=1.5)
    ax.axvline(opt_lr, color='red', linestyle='--', label=f'LR={opt_lr:.1e}')
    ax.set_xscale('log'); ax.set_xlabel('Learning Rate'); ax.set_ylabel('Loss')
    ax.set_title(f'{CNN_MODELS[model_key]}\nOptimal: {opt_lr:.1e}')
    ax.legend(fontsize=9)
    finder.reset()
    del m_tmp, opt_tmp
    torch.cuda.empty_cache()

# LR Finder cho CLIP
print('LR Finder: CLIP...')
clip_lr_ds_tmp  = CLIPDataset(random.sample(train_data, min(1000, len(train_data))), clip_processor)
clip_lr_loader_tmp = DataLoader(clip_lr_ds_tmp, batch_size=32, shuffle=True, num_workers=0)

from transformers import CLIPModel as _CLIPModel

class _TmpCLIP(nn.Module):
    def __init__(self):
        super().__init__()
        base = _CLIPModel.from_pretrained(CLIP_MODEL_NAME)
        self.vision = base.vision_model
        self.proj   = base.visual_projection
        self.head   = nn.Linear(512, NUM_CLASSES)
    def forward(self, x):
        f = self.vision(x).pooler_output
        f = self.proj(f)
        return self.head(F.normalize(f, dim=-1))

clip_tmp = _TmpCLIP().to(DEVICE)
opt_clip_tmp = torch.optim.AdamW(clip_tmp.parameters(), lr=LR_SEARCH_START, weight_decay=1e-4)
finder_clip  = LRFinder(clip_tmp, opt_clip_tmp, crit_sub, device=DEVICE)
finder_clip.range_test(clip_lr_loader_tmp, start_lr=LR_SEARCH_START, end_lr=0.01,
                       num_iter=100, step_mode='exp')
lrs_c   = finder_clip.history['lr']
loss_c  = finder_clip.history['loss']
smooth_c = np.convolve(loss_c, np.ones(5)/5, mode='valid')
best_ic  = int(np.gradient(smooth_c).argmin()) + 2
best_ic  = min(best_ic, len(lrs_c) - 1)
opt_lr_c = lrs_c[best_ic] / 10
OPTIMAL_LRS['clip'] = opt_lr_c

axes_lr[3].plot(lrs_c, loss_c, color=lr_colors[3], lw=1.5)
axes_lr[3].axvline(opt_lr_c, color='red', linestyle='--', label=f'LR={opt_lr_c:.1e}')
axes_lr[3].set_xscale('log'); axes_lr[3].set_xlabel('Learning Rate')
axes_lr[3].set_title(f'CLIP Fine-tuned\nOptimal: {opt_lr_c:.1e}')
axes_lr[3].legend(fontsize=9)
finder_clip.reset()
del clip_tmp, opt_clip_tmp, _TmpCLIP
torch.cuda.empty_cache()

plt.suptitle('LR Finder — 4 Model (steepest-descent heuristic)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'lr_finder.png', bbox_inches='tight')
plt.show()

print('\nOptimal LRs:')
for k, v in OPTIMAL_LRS.items():
    print(f'  {k:30s}: {v:.2e}')
print('Saved: lr_finder.png')

In [ ]:
# ════════════════════════════════════════════════════════════
# BƯỚC 7: ĐỊNH NGHĨA CLIP CLASSIFIER + HÀM TRAIN CLIP
# ════════════════════════════════════════════════════════════

class CLIPClassifier(nn.Module):
    """CLIP ViT image encoder + linear classification head."""
    def __init__(self, clip_model, num_classes=3):
        super().__init__()
        self.clip = clip_model
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, pixel_values):
        feats = self.clip.get_image_features(pixel_values=pixel_values)
        feats = F.normalize(feats, dim=-1)
        return self.classifier(feats)


def train_clip_model(optimal_lr):
    print(f'\n{"="*60}')
    print(f'Training: CLIP Fine-tuned  (lr={optimal_lr:.2e})')
    print(f'{"="*60}')

    base_clip = CLIPModel.from_pretrained(CLIP_MODEL_NAME)
    model = CLIPClassifier(base_clip, num_classes=NUM_CLASSES)
    if torch.cuda.device_count() > 1:
        print(f'  DataParallel: {torch.cuda.device_count()} GPUs')
        model = nn.DataParallel(model)
    model = model.to(DEVICE)

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
    print(f'  Trainable params: {n_params:.1f}M')

    lc  = Counter(lbl for _, lbl in train_data)
    tot = len(train_data)
    cw  = torch.tensor([tot / (NUM_CLASSES * max(lc[i], 1)) for i in range(NUM_CLASSES)],
                       dtype=torch.float).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=cw)
    print(f'  Weights: {[round(float(x), 3) for x in cw]}')

    optimizer = torch.optim.AdamW(model.parameters(), lr=optimal_lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    best_val_f1 = 0.0
    patience_cnt = 0
    history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}
    best_path = OUTPUT_DIR / 'clip_best.pt'

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        model.train()
        tr_loss = 0.0
        for imgs, labels in clip_train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward(); optimizer.step()
            tr_loss += loss.item()
        tr_loss /= len(clip_train_loader)

        model.eval()
        val_loss = 0.0
        preds_v, lbls_v = [], []
        with torch.no_grad():
            for imgs, labels in clip_val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                logits = model(imgs)
                val_loss += criterion(logits, labels).item()
                preds_v.extend(logits.argmax(1).cpu().numpy())
                lbls_v.extend(labels.cpu().numpy())
        val_loss /= len(clip_val_loader)
        val_acc = accuracy_score(lbls_v, preds_v) * 100
        val_f1  = f1_score(lbls_v, preds_v, average='macro') * 100
        scheduler.step()

        elapsed = time.time() - t0
        print(f'  Epoch {epoch:2d}/{EPOCHS} | loss={tr_loss:.4f} | val_loss={val_loss:.4f} | '
              f'acc={val_acc:.2f}% | f1={val_f1:.2f}% | {elapsed:.0f}s')

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            patience_cnt = 0
            raw = model.module if isinstance(model, nn.DataParallel) else model
            torch.save(raw.state_dict(), best_path)
        else:
            patience_cnt += 1
            if patience_cnt >= PATIENCE:
                print(f'  Early stopping at epoch {epoch}')
                break

    # Test evaluation
    print(f'\n  Load best CLIP (val_f1={best_val_f1:.2f}%)...')
    base_eval = CLIPModel.from_pretrained(CLIP_MODEL_NAME)
    eval_model = CLIPClassifier(base_eval, num_classes=NUM_CLASSES)
    eval_model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    eval_model = eval_model.to(DEVICE)
    eval_model.eval()

    preds_t, lbls_t = [], []
    with torch.no_grad():
        for imgs, labels in clip_test_loader:
            imgs = imgs.to(DEVICE)
            preds_t.extend(eval_model(imgs).argmax(1).cpu().numpy())
            lbls_t.extend(labels.numpy())

    test_acc = accuracy_score(lbls_t, preds_t) * 100
    test_f1  = f1_score(lbls_t, preds_t, average='macro') * 100
    print(f'  TEST acc={test_acc:.2f}%  macro_f1={test_f1:.2f}%')
    print(classification_report(lbls_t, preds_t, target_names=CLASS_NAMES))

    del eval_model, base_eval
    return {
        'model_name' : 'CLIP (fine-tuned)',
        'model_key'  : 'clip',
        'best_val_f1': best_val_f1,
        'test_acc'   : test_acc,
        'test_f1'    : test_f1,
        'history'    : history,
        'test_preds' : preds_t,
        'test_labels': lbls_t,
        'best_path'  : str(best_path),
        'is_clip'    : True,
    }

print('CLIPClassifier + train_clip_model định nghĩa xong.')

In [ ]:
# ════════════════════════════════════════════════════════════
# HÀM TRAIN CNN/ViT (EfficientNet, ResNet, ViT)
# optimal_lr được truyền từ LR Finder — không hard-code
# ════════════════════════════════════════════════════════════

def train_one_model(model_key, model_name, optimal_lr):
    print(f'\n{"="*60}')
    print(f'Training: {model_name}  ({model_key})  lr={optimal_lr:.2e}')
    print(f'{"="*60}')

    model = timm.create_model(model_key, pretrained=True, num_classes=NUM_CLASSES)
    if torch.cuda.device_count() > 1:
        print(f'  DataParallel: {torch.cuda.device_count()} GPUs')
        model = nn.DataParallel(model)
    model = model.to(DEVICE)

    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f'  Parameters: {n_params:.1f}M')

    # Weighted loss dựa trên phân phối thực tế trong train set
    label_count = Counter(lbl for _, lbl in train_data)
    total_train = len(train_data)
    class_weights = torch.tensor([
        total_train / (NUM_CLASSES * max(label_count[i], 1))
        for i in range(NUM_CLASSES)
    ], dtype=torch.float).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    print(f'  Class weights: {[round(float(x), 3) for x in class_weights]}')

    optimizer = torch.optim.AdamW(model.parameters(), lr=optimal_lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    best_val_f1  = 0.0
    patience_cnt = 0
    history      = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}
    best_path    = OUTPUT_DIR / f'{model_key}_best.pt'

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        model.train()
        train_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        model.eval()
        val_loss = 0.0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                logits = model(imgs)
                val_loss += criterion(logits, labels).item()
                all_preds.extend(logits.argmax(1).cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        val_loss /= len(val_loader)
        val_acc = accuracy_score(all_labels, all_preds) * 100
        val_f1  = f1_score(all_labels, all_preds, average='macro') * 100
        scheduler.step()

        elapsed = time.time() - t0
        print(f'  Epoch {epoch:2d}/{EPOCHS} | loss={train_loss:.4f} | val_loss={val_loss:.4f} | '
              f'acc={val_acc:.2f}% | f1(macro)={val_f1:.2f}% | {elapsed:.0f}s')

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)

        if val_f1 > best_val_f1:
            best_val_f1  = val_f1
            patience_cnt = 0
            raw_model = model.module if isinstance(model, nn.DataParallel) else model
            torch.save(raw_model.state_dict(), best_path)
        else:
            patience_cnt += 1
            if patience_cnt >= PATIENCE:
                print(f'  Early stopping tại epoch {epoch}')
                break

    # Đánh giá trên test set với best checkpoint
    print(f'\n  Load best (val_f1={best_val_f1:.2f}%)...')
    raw_model = timm.create_model(model_key, pretrained=False, num_classes=NUM_CLASSES)
    raw_model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    raw_model = raw_model.to(DEVICE)
    raw_model.eval()

    test_preds, test_labels = [], []
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs = imgs.to(DEVICE)
            test_preds.extend(raw_model(imgs).argmax(1).cpu().numpy())
            test_labels.extend(labels.numpy())

    test_acc = accuracy_score(test_labels, test_preds) * 100
    test_f1  = f1_score(test_labels, test_preds, average='macro') * 100
    print(f'  TEST acc={test_acc:.2f}%  macro_f1={test_f1:.2f}%')
    print(classification_report(test_labels, test_preds, target_names=CLASS_NAMES))

    del raw_model
    return {
        'model_name' : model_name,
        'model_key'  : model_key,
        'best_val_f1': best_val_f1,
        'test_acc'   : test_acc,
        'test_f1'    : test_f1,
        'history'    : history,
        'test_preds' : test_preds,
        'test_labels': test_labels,
        'best_path'  : str(best_path),
    }

print('Hàm train_one_model đã sẵn sàng.')

In [ ]:
# ════════════════════════════════════════════════════════════
# BƯỚC 8: TRAIN TẤT CẢ 4 MODEL
# ════════════════════════════════════════════════════════════

# ── 3 CNN/ViT ─────────────────────────────────────────────
for model_key, model_name in CNN_MODELS.items():
    result = train_one_model(model_key, model_name, OPTIMAL_LRS[model_key])
    all_results[model_key] = result
    torch.cuda.empty_cache()

# ── CLIP fine-tuned ────────────────────────────────────────
clip_result = train_clip_model(OPTIMAL_LRS['clip'])
all_results['clip'] = clip_result
torch.cuda.empty_cache()

print('\n✓ Đã train xong 4 model.')
print('Keys:', list(all_results.keys()))

In [ ]:
# ════════════════════════════════════════════════════════════
# BẢNG SO SÁNH KẾT QUẢ — Acc / Macro-F1 / VIOLENCE-F1 / AUC / ECE / Inference / Size
# ════════════════════════════════════════════════════════════
import math, time as _time

COLORS_MODEL = {
    'clip_zeroshot'       : '#9E9E9E',
    'efficientnet_b0'     : '#2196F3',
    'resnet50'            : '#4CAF50',
    'vit_base_patch16_224': '#FF5722',
    'clip'                : '#9C27B0',
}

params_map = {
    'clip_zeroshot'       : '~150M',
    'efficientnet_b0'     : '~5M',
    'resnet50'            : '~25M',
    'vit_base_patch16_224': '~86M',
    'clip'                : '~150M',
}

# ── Đo inference time (ms/image) và model size (MB) ───────
def measure_inference_ms(result, n_warmup=5, n_bench=50):
    """Đo latency trên 1 ảnh (batch size = 1)."""
    r = result
    if r.get('is_clip'):
        base = CLIPModel.from_pretrained(CLIP_MODEL_NAME)
        m = CLIPClassifier(base, num_classes=NUM_CLASSES)
        m.load_state_dict(torch.load(r['best_path'], map_location='cpu'))
        # Dummy CLIP input
        dummy = clip_processor(images=Image.new('RGB', (224,224)), return_tensors='pt')
        x_dummy = dummy['pixel_values'].to(DEVICE)
    else:
        m = timm.create_model(r['model_key'], pretrained=False, num_classes=NUM_CLASSES)
        m.load_state_dict(torch.load(r['best_path'], map_location='cpu'))
        x_dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)

    m = m.to(DEVICE).eval()
    with torch.no_grad():
        for _ in range(n_warmup): m(x_dummy)  # warm up
        if torch.cuda.is_available(): torch.cuda.synchronize()
        t0 = _time.perf_counter()
        for _ in range(n_bench): m(x_dummy)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        elapsed_ms = (_time.perf_counter() - t0) / n_bench * 1000

    import os
    size_mb = os.path.getsize(r['best_path']) / 1024**2
    del m; torch.cuda.empty_cache()
    return round(elapsed_ms, 2), round(size_mb, 1)

print('Đo inference time + model size...')
for key in list(CNN_MODELS.keys()) + ['clip']:
    if key not in all_results: continue
    ms, mb = measure_inference_ms(all_results[key])
    all_results[key]['infer_ms'] = ms
    all_results[key]['size_mb']  = mb
    print(f'  {all_results[key]["model_name"]:26s}: {ms:6.2f}ms/img  {mb:6.1f}MB')

# ── Per-class F1 (lấy VIOLENCE-F1 riêng) ─────────────────
for key in list(CNN_MODELS.keys()) + ['clip']:
    if key not in all_results: continue
    r = all_results[key]
    from sklearn.metrics import f1_score as _f1
    per_class = _f1(r['test_labels'], r['test_preds'], average=None, labels=[0,1,2])
    r['violence_f1'] = float(per_class[LABEL_VIOLENCE] * 100)

# ── In bảng ────────────────────────────────────────────────
H = f'{"Model":<26} {"Acc":>7} {"MacroF1":>9} {"VIO-F1":>8} {"AUC":>7} {"ECE":>7} {"ms/img":>8} {"MB":>6}'
print('\n' + '='*len(H))
print(H)
print('='*len(H))

# CLIP zero-shot
zs = all_results.get('clip_zeroshot', {})
if zs:
    from sklearn.metrics import f1_score as _f1
    vio_f1_zs = _f1(zs['test_labels'], zs['test_preds'], average=None)[LABEL_VIOLENCE]*100
    zs['violence_f1'] = float(vio_f1_zs)
    print(f'{"CLIP (zero-shot)":<26} {zs["test_acc"]:>6.2f}% {zs["test_f1"]:>8.2f}% '
          f'{vio_f1_zs:>7.2f}%    {"—":>5}   {"—":>5}       {"—":>5}    {"—":>4}')

for key in ['efficientnet_b0', 'resnet50', 'vit_base_patch16_224', 'clip']:
    if key not in all_results: continue
    r = all_results[key]
    auc_s = f'{r["roc_auc"]:.4f}' if 'roc_auc' in r else '   —'
    ece_s = f'{r["ece"]:.4f}' if 'ece' in r else '   —'
    ms_s  = f'{r["infer_ms"]:.1f}ms' if 'infer_ms' in r else '    —'
    mb_s  = f'{r["size_mb"]:.0f}' if 'size_mb' in r else ' —'
    print(f'{r["model_name"]:<26} {r["test_acc"]:>6.2f}% {r["test_f1"]:>8.2f}% '
          f'{r["violence_f1"]:>7.2f}% {auc_s:>7} {ece_s:>7} {ms_s:>8} {mb_s:>5}MB')

print('='*len(H))

trained_keys = [k for k in all_results if k != 'clip_zeroshot']
best_key = max(trained_keys, key=lambda k: all_results[k]['test_f1'])
print(f'\n★ Tốt nhất (Macro F1): {all_results[best_key]["model_name"]} — '
      f'F1={all_results[best_key]["test_f1"]:.2f}%')

# Lưu JSON đầy đủ
summary = {}
for k, v in all_results.items():
    summary[k] = {kk: vv for kk, vv in v.items()
                  if kk not in ('history', 'test_preds', 'test_labels')}
with open(OUTPUT_DIR / 'comparison_results.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print('Saved: comparison_results.json')

In [ ]:
# ════════════════════════════════════════════════════════════
# TRAINING CURVES — Loss + Macro F1 cho 4 model
# ════════════════════════════════════════════════════════════
trained_keys = [k for k in ['efficientnet_b0', 'resnet50', 'vit_base_patch16_224', 'clip']
                if k in all_results and 'history' in all_results[k]]
n_models = len(trained_keys)

fig, axes_c = plt.subplots(2, n_models, figsize=(5*n_models, 8))

for col, mkey in enumerate(trained_keys):
    r = all_results[mkey]
    h = r['history']
    ep = range(1, len(h['train_loss']) + 1)
    c  = COLORS_MODEL.get(mkey, '#607D8B')

    ax_loss = axes_c[0][col]
    ax_loss.plot(ep, h['train_loss'], '--', color=c, alpha=0.6, label='Train loss')
    ax_loss.plot(ep, h['val_loss'],   '-',  color=c, lw=2,       label='Val loss')
    ax_loss.set_title(f'{r["model_name"]}', fontsize=11, fontweight='bold')
    ax_loss.set_xlabel('Epoch'); ax_loss.set_ylabel('Loss')
    ax_loss.legend(fontsize=8)

    ax_f1 = axes_c[1][col]
    ax_f1.plot(ep, h['val_f1'], '-', color=c, lw=2,
               label=f'Val Macro F1 (best={r["best_val_f1"]:.1f}%)')
    ax_f1.axhline(r['best_val_f1'], linestyle=':', color=c, alpha=0.4)
    ax_f1.set_xlabel('Epoch'); ax_f1.set_ylabel('Macro F1 (%)')
    ax_f1.set_ylim(0, 100)
    ax_f1.legend(fontsize=8)

plt.suptitle('Training Curves — 4 Model (Macro F1 = tiêu chí chọn best)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: training_curves.png')

In [ ]:
# ════════════════════════════════════════════════════════════
# CONFUSION MATRICES — 4 model (2×2 grid)
# ════════════════════════════════════════════════════════════
all_trained = [k for k in ['efficientnet_b0', 'resnet50', 'vit_base_patch16_224', 'clip']
               if k in all_results]
n = len(all_trained)
cols_cm = min(4, n)
rows_cm = math.ceil(n / cols_cm)

fig, axes_cm = plt.subplots(rows_cm, cols_cm, figsize=(5*cols_cm, 4.5*rows_cm))
axes_cm_flat = axes_cm.flatten() if n > 1 else [axes_cm]

for ax, mkey in zip(axes_cm_flat, all_trained):
    r = all_results[mkey]
    cm = confusion_matrix(r['test_labels'], r['test_preds'])
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
    sns.heatmap(cm_pct, annot=True, fmt='.1f', ax=ax,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                cmap='Blues', cbar=False)
    ax.set_title(f'{r["model_name"]}\nAcc={r["test_acc"]:.1f}%  F1={r["test_f1"]:.1f}%',
                 fontsize=10)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')

for j in range(n, len(axes_cm_flat)): axes_cm_flat[j].axis('off')

plt.suptitle('Confusion Matrices (%) — 4 Model', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrices.png')

In [ ]:
# ════════════════════════════════════════════════════════════
# ĐÁNH GIÁ: ROC CURVES + PRECISION-RECALL CURVES + ECE
# ════════════════════════════════════════════════════════════

def load_model_for_eval(result):
    """Load best checkpoint, trả về model ready for inference."""
    r = result
    if r.get('is_clip'):
        base = CLIPModel.from_pretrained(CLIP_MODEL_NAME)
        m = CLIPClassifier(base, num_classes=NUM_CLASSES)
        m.load_state_dict(torch.load(r['best_path'], map_location='cpu'))
    else:
        m = timm.create_model(r['model_key'], pretrained=False, num_classes=NUM_CLASSES)
        m.load_state_dict(torch.load(r['best_path'], map_location='cpu'))
    return m.to(DEVICE).eval()

def get_softmax_probs(result):
    """Trả về (probs: np.ndarray [N,3], labels: np.ndarray [N])."""
    r = result
    loader = clip_test_loader if r.get('is_clip') else test_loader
    m = load_model_for_eval(r)
    probs_out, labels_out = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs = imgs.to(DEVICE)
            p = torch.softmax(m(imgs), dim=1).cpu().numpy()
            probs_out.extend(p)
            labels_out.extend(lbls.numpy())
    del m; torch.cuda.empty_cache()
    return np.array(probs_out), np.array(labels_out)

# ── Kéo probabilities cho từng model ─────────────────────
print('Tính softmax probabilities cho 4 models...')
model_probs = {}
eval_keys = [k for k in ['efficientnet_b0', 'resnet50', 'vit_base_patch16_224', 'clip']
             if k in all_results]
for mkey in eval_keys:
    print(f'  {all_results[mkey]["model_name"]}...')
    probs, lbls = get_softmax_probs(all_results[mkey])
    model_probs[mkey] = {'probs': probs, 'labels': lbls}
    auc = roc_auc_score(lbls, probs, multi_class='ovr', average='macro')
    all_results[mkey]['roc_auc'] = float(auc)
    print(f'    AUC-ROC macro OvR: {auc:.4f}')

# ── ROC Curves (per class × model) ───────────────────────
n_ev = len(eval_keys)
fig, axes_roc = plt.subplots(NUM_CLASSES, n_ev, figsize=(4.5*n_ev, 4*NUM_CLASSES))
for col, mkey in enumerate(eval_keys):
    mp = model_probs[mkey]
    yb = np.eye(NUM_CLASSES)[mp['labels']]
    for row, cname in enumerate(CLASS_NAMES):
        ax = axes_roc[row][col] if n_ev > 1 else axes_roc[row]
        fpr, tpr, _ = roc_curve(yb[:, row], mp['probs'][:, row])
        auc_c = roc_auc_score(yb[:, row], mp['probs'][:, row])
        c = COLORS_MODEL.get(mkey, '#607D8B')
        ax.plot(fpr, tpr, color=c, lw=2, label=f'AUC={auc_c:.3f}')
        ax.plot([0,1],[0,1],'--', color='gray', alpha=0.4)
        ax.set_title(f'{all_results[mkey]["model_name"]}\n{cname}', fontsize=9)
        ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
        ax.legend(fontsize=8)
plt.suptitle('ROC Curves (One-vs-Rest)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig(OUTPUT_DIR/'roc_curves.png', bbox_inches='tight'); plt.show()

# ── PR Curves (per class × model) ────────────────────────
fig, axes_pr = plt.subplots(NUM_CLASSES, n_ev, figsize=(4.5*n_ev, 4*NUM_CLASSES))
for col, mkey in enumerate(eval_keys):
    mp = model_probs[mkey]
    yb = np.eye(NUM_CLASSES)[mp['labels']]
    for row, cname in enumerate(CLASS_NAMES):
        ax = axes_pr[row][col] if n_ev > 1 else axes_pr[row]
        prec, rec, _ = precision_recall_curve(yb[:, row], mp['probs'][:, row])
        ap = average_precision_score(yb[:, row], mp['probs'][:, row])
        base = yb[:, row].mean()
        c = COLORS_MODEL.get(mkey, '#607D8B')
        ax.plot(rec, prec, color=c, lw=2, label=f'AP={ap:.3f}')
        ax.axhline(base, linestyle='--', color='gray', alpha=0.4, label=f'base={base:.2f}')
        ax.set_title(f'{all_results[mkey]["model_name"]}\n{cname}', fontsize=9)
        ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
        ax.legend(fontsize=8)
plt.suptitle('Precision-Recall Curves', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig(OUTPUT_DIR/'pr_curves.png', bbox_inches='tight'); plt.show()

# ── ECE Calibration + Reliability Diagram ────────────────
def compute_ece(probs, labels, n_bins=15):
    conf_arr = probs.max(axis=1)
    pred_arr = probs.argmax(axis=1)
    corr_arr = (pred_arr == labels).astype(float)
    edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0; bin_accs, bin_confs = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (conf_arr >= lo) & (conf_arr < hi)
        if mask.sum() == 0:
            bin_accs.append(0.); bin_confs.append((lo+hi)/2); continue
        a = corr_arr[mask].mean(); c = conf_arr[mask].mean()
        ece += mask.sum() / len(labels) * abs(a - c)
        bin_accs.append(a); bin_confs.append(c)
    return float(ece), bin_accs, bin_confs

fig, axes_ece = plt.subplots(1, n_ev, figsize=(4.5*n_ev, 4.5))
for col, mkey in enumerate(eval_keys):
    mp = model_probs[mkey]
    ece, b_accs, b_confs = compute_ece(mp['probs'], mp['labels'])
    all_results[mkey]['ece'] = ece
    ax = axes_ece[col] if n_ev > 1 else axes_ece
    n_b = len(b_accs); centers = np.linspace(1/(2*n_b), 1-1/(2*n_b), n_b)
    c = COLORS_MODEL.get(mkey, '#607D8B')
    ax.bar(centers, b_accs, width=1/n_b, alpha=0.6, color=c, label='Accuracy')
    ax.plot([0,1],[0,1],'--r', lw=1.5, label='Perfect')
    ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.set_xlabel('Confidence'); ax.set_ylabel('Accuracy')
    ax.set_title(f'{all_results[mkey]["model_name"]}\nECE={ece:.4f}', fontsize=10)
    ax.legend(fontsize=8)
plt.suptitle('Reliability Diagram — Confidence Calibration', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig(OUTPUT_DIR/'calibration.png', bbox_inches='tight'); plt.show()

print('\nSaved: roc_curves.png + pr_curves.png + calibration.png')
print('\nECE (thấp hơn = tốt hơn):')
for k in eval_keys:
    print(f'  {all_results[k]["model_name"]:26s}: ECE={all_results[k]["ece"]:.4f}')

In [ ]:
# ════════════════════════════════════════════════════════════
# PHÂN TÍCH LỖI + EXPLAINABILITY
# 1. High-confidence errors (model sai nhưng rất tự tin)
# 2. High-entropy predictions (model không chắc chắn)
# 3. Grad-CAM (EfficientNet, ResNet)
# 4. ViT Attention Rollout
# ════════════════════════════════════════════════════════════

def entropy(p_row):
    p = np.clip(p_row, 1e-9, 1.0)
    return float(-np.sum(p * np.log(p)))

# Dùng best model để phân tích lỗi
best_key_ea = max(eval_keys, key=lambda k: all_results[k]['test_f1'])
mp_ea   = model_probs[best_key_ea]
probs_e = mp_ea['probs']; lbls_e = mp_ea['labels']; preds_e = probs_e.argmax(axis=1)
correct_e = (preds_e == lbls_e)
print(f'Phân tích lỗi: {all_results[best_key_ea]["model_name"]}')

# ── 1. High-confidence errors ─────────────────────────────
wrong_idx = np.where(~correct_e)[0]
if len(wrong_idx) > 0:
    wrong_conf = probs_e[wrong_idx].max(axis=1)
    top_err = wrong_idx[np.argsort(-wrong_conf)[:12]]
    n_show = len(top_err); cols_e = 4; rows_e = math.ceil(n_show/cols_e)
    fig, ax_e = plt.subplots(rows_e, cols_e, figsize=(cols_e*3, rows_e*3))
    ax_e = ax_e.flatten()
    for i, idx in enumerate(top_err):
        img_p, _ = test_data[idx]
        if not isinstance(img_p, Image.Image): img_p = Image.fromarray(img_p)
        ax_e[i].imshow(img_p)
        ax_e[i].set_title(f'True:{CLASS_NAMES[lbls_e[idx]]}\n'
                          f'Pred:{CLASS_NAMES[preds_e[idx]]} ({probs_e[idx].max()*100:.0f}%)',
                          fontsize=7, color='red')
        ax_e[i].axis('off')
    for j in range(i+1, len(ax_e)): ax_e[j].axis('off')
    plt.suptitle('High-confidence errors', fontsize=11, fontweight='bold')
    plt.tight_layout(); plt.savefig(OUTPUT_DIR/'error_high_conf.png', bbox_inches='tight'); plt.show()

# ── 2. High-entropy (uncertain) predictions ──────────────
entropies  = np.array([entropy(p) for p in probs_e])
top_unc = np.argsort(-entropies)[:12]
fig, ax_u = plt.subplots(3, 4, figsize=(12, 9))
for i, idx in enumerate(top_unc):
    img_p, _ = test_data[idx]
    if not isinstance(img_p, Image.Image): img_p = Image.fromarray(img_p)
    ok_str = 'correct' if correct_e[idx] else 'WRONG'
    c_col  = 'green' if correct_e[idx] else 'darkorange'
    ax_u.flat[i].imshow(img_p)
    ax_u.flat[i].set_title(f'True:{CLASS_NAMES[lbls_e[idx]]} [{ok_str}]\n'
                           f'Entropy={entropies[idx]:.3f}', fontsize=7, color=c_col)
    ax_u.flat[i].axis('off')
plt.suptitle('High-entropy (uncertain) predictions', fontsize=11, fontweight='bold')
plt.tight_layout(); plt.savefig(OUTPUT_DIR/'error_uncertain.png', bbox_inches='tight'); plt.show()

# ── 3. Grad-CAM cho CNN models ───────────────────────────
def denorm_img(tensor_img):
    t = tensor_img.clone()
    for c, (m, s) in enumerate(zip(MEAN_USE, STD_USE)):
        t[c] = t[c] * s + m
    return t.clamp(0,1).permute(1,2,0).numpy()

# Lấy 2 ảnh mỗi class để visualize
vis_indices = []
for lbl in range(NUM_CLASSES):
    cands = [i for i,(_, l) in enumerate(test_data) if l==lbl]
    vis_indices += random.sample(cands, min(2, len(cands)))

cnn_gc_keys = [k for k in ['efficientnet_b0', 'resnet50'] if k in all_results]
for mkey in cnn_gc_keys:
    r_gc = all_results[mkey]
    m_gc = load_model_for_eval(r_gc)

    if 'efficientnet' in mkey:
        target_layers = [m_gc.conv_head]
    else:
        target_layers = [m_gc.layer4[-1]]

    cam = GradCAM(model=m_gc, target_layers=target_layers)
    n_v = len(vis_indices)
    fig, ax_gc = plt.subplots(n_v, 2, figsize=(6, n_v*2.5))

    for row, idx in enumerate(vis_indices):
        img_p, true_l = test_data[idx]
        if not isinstance(img_p, Image.Image): img_p = Image.fromarray(img_p)
        inp = val_transform(img_p).unsqueeze(0).to(DEVICE)
        rgb = denorm_img(inp.squeeze(0))
        gcam = cam(input_tensor=inp, targets=[ClassifierOutputTarget(true_l)])[0]
        cam_vis = show_cam_on_image(rgb, gcam, use_rgb=True)
        ax_gc[row][0].imshow(rgb); ax_gc[row][0].set_title(f'{CLASS_NAMES[true_l]}', fontsize=8); ax_gc[row][0].axis('off')
        ax_gc[row][1].imshow(cam_vis); ax_gc[row][1].set_title('Grad-CAM', fontsize=8); ax_gc[row][1].axis('off')

    plt.suptitle(f'Grad-CAM — {r_gc["model_name"]}', fontsize=12, fontweight='bold')
    plt.tight_layout(); plt.savefig(OUTPUT_DIR/f'gradcam_{mkey}.png', bbox_inches='tight'); plt.show()
    del m_gc, cam; torch.cuda.empty_cache()

# ── 4. ViT Attention Rollout ──────────────────────────────
if 'vit_base_patch16_224' in all_results:
    print('ViT Attention Rollout...')
    vit_r = all_results['vit_base_patch16_224']
    vit_m = load_model_for_eval(vit_r)
    attn_maps = []
    hooks = [blk.attn.register_forward_hook(lambda m,i,o: attn_maps.append(o.detach().cpu()))
             for blk in vit_m.blocks]

    n_vv = min(6, len(vis_indices))
    fig, ax_vit = plt.subplots(n_vv, 2, figsize=(6, n_vv*2.5))
    for row, idx in enumerate(vis_indices[:n_vv]):
        attn_maps.clear()
        img_p, true_l = test_data[idx]
        if not isinstance(img_p, Image.Image): img_p = Image.fromarray(img_p)
        inp = val_transform(img_p).unsqueeze(0).to(DEVICE)
        rgb = denorm_img(inp.squeeze(0))
        with torch.no_grad(): vit_m(inp)
        rollout = torch.eye(attn_maps[0].shape[-1])
        for a in attn_maps:
            a_avg = a[0].mean(0)
            a_avg = a_avg + torch.eye(a_avg.shape[0])
            a_avg = a_avg / a_avg.sum(-1, keepdim=True)
            rollout = rollout @ a_avg
        mask = rollout[0, 1:].reshape(14, 14).numpy()
        mask = (mask - mask.min()) / (mask.max() - mask.min() + 1e-8)
        mask_up = np.array(Image.fromarray((mask*255).astype(np.uint8)).resize((224,224))) / 255.0
        ax_vit[row][0].imshow(rgb); ax_vit[row][0].set_title(CLASS_NAMES[true_l], fontsize=8); ax_vit[row][0].axis('off')
        ax_vit[row][1].imshow(rgb); ax_vit[row][1].imshow(mask_up, alpha=0.55, cmap='hot')
        ax_vit[row][1].set_title('Attention Rollout', fontsize=8); ax_vit[row][1].axis('off')

    for h in hooks: h.remove()
    plt.suptitle('ViT Attention Rollout', fontsize=12, fontweight='bold')
    plt.tight_layout(); plt.savefig(OUTPUT_DIR/'vit_attention.png', bbox_inches='tight'); plt.show()
    del vit_m; torch.cuda.empty_cache()

print('Saved: error_high_conf.png + error_uncertain.png + gradcam_*.png + vit_attention.png')

In [ ]:
# ════════════════════════════════════════════════════════════
# t-SNE EMBEDDING VISUALIZATION + RADAR CHART 6 CHIỀU
# ════════════════════════════════════════════════════════════

# ── t-SNE: trích features từ penultimate layer ────────────
def extract_penultimate(result, max_n=800):
    r = result
    is_clip = r.get('is_clip', False)
    loader  = clip_test_loader if is_clip else test_loader

    if is_clip:
        base = CLIPModel.from_pretrained(CLIP_MODEL_NAME)
        m = CLIPClassifier(base, num_classes=NUM_CLASSES)
        m.load_state_dict(torch.load(r['best_path'], map_location='cpu'))
        m = m.to(DEVICE).eval()
        feats, lbls_out = [], []
        with torch.no_grad():
            for imgs, lbs in loader:
                imgs = imgs.to(DEVICE)
                f = m.clip.get_image_features(pixel_values=imgs)
                f = F.normalize(f, dim=-1)
                feats.extend(f.cpu().numpy()); lbls_out.extend(lbs.numpy())
                if len(feats) >= max_n: break
        del m
    else:
        m = timm.create_model(r['model_key'], pretrained=False, num_classes=NUM_CLASSES)
        m.load_state_dict(torch.load(r['best_path'], map_location='cpu'))
        feat_buf = []
        if 'vit' in r['model_key']:
            hook = m.head.register_forward_pre_hook(lambda mo,i,o_: feat_buf.append(i[0].detach().cpu()))
        elif 'efficientnet' in r['model_key']:
            hook = m.classifier.register_forward_pre_hook(lambda mo,i,o_: feat_buf.append(i[0].detach().cpu()))
        else:
            hook = m.fc.register_forward_pre_hook(lambda mo,i,o_: feat_buf.append(i[0].detach().cpu()))
        m = m.to(DEVICE).eval()
        feats, lbls_out = [], []
        with torch.no_grad():
            for imgs, lbs in loader:
                feat_buf.clear(); m(imgs.to(DEVICE))
                f = feat_buf[0]
                if f.dim() > 2: f = f.squeeze(-1).squeeze(-1)
                feats.extend(f.numpy()); lbls_out.extend(lbs.numpy())
                if len(feats) >= max_n: break
        hook.remove(); del m

    torch.cuda.empty_cache()
    return np.array(feats[:max_n]), np.array(lbls_out[:max_n])

print('t-SNE visualization...')
tsne_keys = [k for k in eval_keys if k in all_results]
n_tsne = len(tsne_keys)
fig, ax_ts = plt.subplots(1, n_tsne, figsize=(5.5*n_tsne, 5))
ax_ts = [ax_ts] if n_tsne == 1 else list(ax_ts)
clrs_t = ['#2196F3', '#FF5722', '#4CAF50']
marks_t = ['o', 's', '^']

for ax, mkey in zip(ax_ts, tsne_keys):
    print(f'  {all_results[mkey]["model_name"]}...')
    feats, lbls_t = extract_penultimate(all_results[mkey], max_n=800)
    emb = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=1000).fit_transform(feats)
    for cls_i in range(NUM_CLASSES):
        mask = lbls_t == cls_i
        ax.scatter(emb[mask,0], emb[mask,1], c=clrs_t[cls_i], marker=marks_t[cls_i],
                   s=15, alpha=0.6, label=CLASS_NAMES[cls_i])
    ax.set_title(f'{all_results[mkey]["model_name"]}\nF1={all_results[mkey]["test_f1"]:.1f}%',
                 fontsize=10)
    ax.legend(fontsize=8, markerscale=2); ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('t-SNE Feature Space (test set, 800 samples/model)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig(OUTPUT_DIR/'tsne.png', bbox_inches='tight'); plt.show()

# ── Radar Chart 6 chiều ────────────────────────────────────
# Chiều: Acc / Macro-F1 / VIOLENCE-F1 / AUC-ROC / Calibration(1-ECE) / Speed(1/ms)
metric_labels = [
    'Test Acc',
    'Macro F1',
    'VIOLENCE F1',
    'AUC-ROC',
    '1 - ECE\n(calibration)',
    'Speed\n(1/latency)',
]
N_m = len(metric_labels)
angles = np.linspace(0, 2*np.pi, N_m, endpoint=False).tolist(); angles += angles[:1]

# Normalize speed: 1/ms, then scale to [0,1] relative to fastest model
infer_vals = {k: 1.0 / all_results[k]['infer_ms'] for k in eval_keys if 'infer_ms' in all_results[k]}
max_speed  = max(infer_vals.values()) if infer_vals else 1.0

fig, ax_r = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
palette = plt.cm.Set2(np.linspace(0, 1, len(eval_keys)))

for color, mkey in zip(palette, eval_keys):
    r = all_results[mkey]
    speed_norm = (infer_vals.get(mkey, 0) / max_speed) if max_speed > 0 else 0
    vals = [
        r['test_acc']        / 100,
        r['test_f1']         / 100,
        r.get('violence_f1', 0) / 100,
        r.get('roc_auc', 0.5),
        1.0 - min(r.get('ece', 0.15), 0.5),
        speed_norm,
    ]; vals += vals[:1]
    ax_r.plot(angles, vals, lw=2.5, label=r['model_name'], color=color)
    ax_r.fill(angles, vals, alpha=0.10, color=color)

ax_r.set_xticks(angles[:-1]); ax_r.set_xticklabels(metric_labels, fontsize=10)
ax_r.set_ylim(0, 1); ax_r.set_yticks([0.2,0.4,0.6,0.8,1.0])
ax_r.set_yticklabels(['0.2','0.4','0.6','0.8','1.0'], fontsize=7)
ax_r.legend(loc='upper right', bbox_to_anchor=(1.45, 1.15), fontsize=10)
ax_r.set_title('Radar Chart 6 chiều — So sánh 4 Model', fontsize=14, fontweight='bold', pad=25)
plt.tight_layout(); plt.savefig(OUTPUT_DIR/'radar_chart.png', bbox_inches='tight'); plt.show()

print('Saved: tsne.png + radar_chart.png')

In [ ]:
# ════════════════════════════════════════════════════════════
# KẾT LUẬN CÓ CĂN CỨ: MODEL NÀO DÙNG CHO PRODUCTION?
# Phân tích trade-off dựa trên số liệu thực nghiệm
# ════════════════════════════════════════════════════════════
print('=' * 68)
print('  KẾT LUẬN THỰC NGHIỆM')
print('=' * 68)

# ── Thu thập số liệu ─────────────────────────────────────
results_summary = []
for key in ['efficientnet_b0', 'resnet50', 'vit_base_patch16_224', 'clip']:
    if key not in all_results: continue
    r = all_results[key]
    results_summary.append({
        'key'         : key,
        'name'        : r['model_name'],
        'test_f1'     : r['test_f1'],
        'test_acc'    : r['test_acc'],
        'violence_f1' : r.get('violence_f1', 0),
        'roc_auc'     : r.get('roc_auc', 0),
        'ece'         : r.get('ece', 1),
        'infer_ms'    : r.get('infer_ms', 999),
        'size_mb'     : r.get('size_mb', 999),
    })

if not results_summary:
    print('[!] Chưa có kết quả training.')
else:
    # Sắp xếp theo Macro F1
    results_summary.sort(key=lambda x: x['test_f1'], reverse=True)

    best_overall = results_summary[0]
    best_speed   = min(results_summary, key=lambda x: x['infer_ms'])
    best_violence= max(results_summary, key=lambda x: x['violence_f1'])
    best_calib   = min(results_summary, key=lambda x: x['ece'])

    print(f'\n📊 Tổng quan:')
    print(f'   Model tốt nhất (Macro F1)  : {best_overall["name"]:26s} F1={best_overall["test_f1"]:.2f}%')
    print(f'   Model nhanh nhất           : {best_speed["name"]:26s} {best_speed["infer_ms"]:.1f}ms/img')
    print(f'   VIOLENCE-F1 cao nhất       : {best_violence["name"]:26s} {best_violence["violence_f1"]:.2f}%')
    print(f'   Calibrated nhất (ECE thấp) : {best_calib["name"]:26s} ECE={best_calib["ece"]:.4f}')

    print(f'\n🏭 Khuyến nghị production:')

    # Logic lựa chọn dựa trên kết quả thực
    prod_key = best_overall['key']
    prod_r   = best_overall

    # Nếu EfficientNet-B0 chênh lệch F1 ≤ 1.5% so với best nhưng nhanh hơn nhiều → ưu tiên
    eff_r = next((x for x in results_summary if x['key'] == 'efficientnet_b0'), None)
    if eff_r and (best_overall['test_f1'] - eff_r['test_f1']) <= 1.5 and \
       eff_r['infer_ms'] < best_overall['infer_ms'] * 0.7:
        prod_key = 'efficientnet_b0'
        prod_r   = eff_r
        print(f'   → EfficientNet-B0 được chọn vì chênh F1 chỉ '
              f'{best_overall["test_f1"] - eff_r["test_f1"]:.1f}% so với best')
        print(f'     nhưng nhanh hơn {best_overall["infer_ms"]/eff_r["infer_ms"]:.1f}× '
              f'({eff_r["infer_ms"]:.1f}ms vs {best_overall["infer_ms"]:.1f}ms)')
    else:
        print(f'   → {prod_r["name"]} được chọn vì đạt Macro F1 cao nhất ({prod_r["test_f1"]:.2f}%)')

    print(f'\n   Model production: {prod_r["name"]}')
    print(f'     - Test Acc       : {prod_r["test_acc"]:.2f}%')
    print(f'     - Macro F1       : {prod_r["test_f1"]:.2f}%')
    print(f'     - VIOLENCE F1    : {prod_r["violence_f1"]:.2f}%')
    print(f'     - AUC-ROC        : {prod_r["roc_auc"]:.4f}')
    print(f'     - ECE            : {prod_r["ece"]:.4f}')
    print(f'     - Latency        : {prod_r["infer_ms"]:.1f}ms/image')
    print(f'     - Size           : {prod_r["size_mb"]:.0f}MB')

    print(f'\n   Lý do chọn cho hệ thống blog platform:')
    print(f'     1. Macro F1 đảm bảo không bỏ sót VIOLENCE (class thiểu số quan trọng)')
    print(f'     2. AUC-ROC cao → phân biệt tốt giữa các class ở mọi threshold')
    print(f'     3. ECE thấp → confidence score đáng tin cậy để set ngưỡng kiểm duyệt')
    print(f'     4. Latency {prod_r["infer_ms"]:.0f}ms/img → đủ nhanh cho upload pipeline')

    print(f'\n   Lưu ý triển khai:')
    print(f'     - Dùng threshold > 0.6 (thay vì argmax) để reduce false positive trên SAFE')
    print(f'     - CLIP zero-shot chỉ đạt ~{all_results["clip_zeroshot"]["test_f1"]:.0f}% F1 → fine-tuning là bắt buộc')
    print(f'     - Model đã lưu tại: best_image_model.zip (production-ready)')

    # Lưu recommendation vào JSON
    recommendation = {
        'production_model'     : prod_key,
        'production_model_name': prod_r['name'],
        'selection_reason'     : 'highest_macro_f1' if prod_key == best_overall['key'] else 'speed_quality_tradeoff',
        'metrics'              : prod_r,
        'zeroshot_baseline_f1' : all_results.get('clip_zeroshot', {}).get('test_f1', 0),
        'all_models_ranked'    : [{'rank': i+1, **x} for i, x in enumerate(results_summary)],
    }
    with open(OUTPUT_DIR / 'production_recommendation.json', 'w') as f:
        json.dump(recommendation, f, indent=2, default=str)
    print('\nSaved: production_recommendation.json')

print('=' * 68)

In [ ]:
# ════════════════════════════════════════════════════════════
# LƯU MODEL TỐT NHẤT (theo Macro F1 trên test)
# ════════════════════════════════════════════════════════════
import shutil

trained_keys_save = [k for k in all_results if k != 'clip_zeroshot']
best_key_save = max(trained_keys_save, key=lambda k: all_results[k]['test_f1'])
best_info_save = all_results[best_key_save]
print(f'Model tốt nhất: {best_info_save["model_name"]}')
print(f'  Test Acc: {best_info_save["test_acc"]:.2f}%  Macro F1: {best_info_save["test_f1"]:.2f}%')

best_dir = OUTPUT_DIR / 'best_image_model'
best_dir.mkdir(exist_ok=True)
shutil.copy(best_info_save['best_path'], best_dir / 'model.pt')

meta = {
    'model_key'   : best_key_save,
    'model_name'  : best_info_save['model_name'],
    'is_clip'     : best_info_save.get('is_clip', False),
    'num_classes' : NUM_CLASSES,
    'class_names' : CLASS_NAMES,
    'img_size'    : IMG_SIZE,
    'mean'        : MEAN_USE,
    'std'         : STD_USE,
    'test_acc'    : best_info_save['test_acc'],
    'test_f1'     : best_info_save['test_f1'],
    'roc_auc'     : best_info_save.get('roc_auc'),
    'ece'         : best_info_save.get('ece'),
    'all_models'  : {k: {kk: vv for kk, vv in v.items()
                         if kk not in ('history', 'test_preds', 'test_labels')}
                    for k, v in all_results.items()},
}
with open(best_dir / 'meta.json', 'w') as f:
    json.dump(meta, f, indent=2, default=str)

shutil.make_archive(str(OUTPUT_DIR / 'best_image_model'), 'zip', str(best_dir))
zip_mb = os.path.getsize(str(OUTPUT_DIR / 'best_image_model.zip')) / 1024**2
print(f'Saved: best_image_model.zip ({zip_mb:.1f} MB)')
print('→ Vào tab Output của Kaggle để download file .zip')

In [ ]:
# ════════════════════════════════════════════════════════════
# INFERENCE DEMO
# ════════════════════════════════════════════════════════════

# Load best model (auto-detect CNN/ViT vs CLIP)
if best_info_save.get('is_clip'):
    _base = CLIPModel.from_pretrained(CLIP_MODEL_NAME)
    infer_model = CLIPClassifier(_base, num_classes=NUM_CLASSES)
    infer_model.load_state_dict(torch.load(best_dir / 'model.pt', map_location=DEVICE))
    def _infer_tf(img): return clip_processor(images=img, return_tensors='pt')['pixel_values'].squeeze(0)
else:
    infer_model = timm.create_model(best_key_save, pretrained=False, num_classes=NUM_CLASSES)
    infer_model.load_state_dict(torch.load(best_dir / 'model.pt', map_location=DEVICE))
    def _infer_tf(img): return val_transform(img)

infer_model = infer_model.to(DEVICE)
infer_model.eval()

def predict_image(img_input):
    if isinstance(img_input, str):
        img = Image.open(img_input).convert('RGB')
    elif isinstance(img_input, Image.Image):
        img = img_input.convert('RGB')
    else:
        img = Image.fromarray(img_input).convert('RGB')
    tensor = _infer_tf(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(infer_model(tensor), dim=1)[0].cpu().numpy()
    pred = int(probs.argmax())
    return {
        'label'      : CLASS_NAMES[pred],
        'confidence' : float(probs[pred]),
        'scores'     : {CLASS_NAMES[i]: round(float(p), 4) for i, p in enumerate(probs)},
    }

# Demo 6 ảnh (2 mỗi class) từ test set
print(f'Demo inference — {best_info_save["model_name"]}\n')
demo_idxs = []
for lbl in range(NUM_CLASSES):
    cands = [i for i, (_, l) in enumerate(test_data) if l == lbl]
    demo_idxs += random.sample(cands, min(2, len(cands)))

fig, axes_d = plt.subplots(2, 3, figsize=(12, 7))
for i, idx in enumerate(demo_idxs):
    img_pil, true_lbl = test_data[idx]
    if not isinstance(img_pil, Image.Image): img_pil = Image.fromarray(img_pil)
    res = predict_image(img_pil)
    ok  = res['label'] == CLASS_NAMES[true_lbl]
    axes_d.flat[i].imshow(img_pil)
    axes_d.flat[i].set_title(
        f'True: {CLASS_NAMES[true_lbl]}\nPred: {res["label"]} ({res["confidence"]*100:.1f}%)',
        color='green' if ok else 'red', fontsize=9)
    axes_d.flat[i].axis('off')

plt.suptitle(f'Inference Demo — {best_info_save["model_name"]}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'inference_demo.png', bbox_inches='tight')
plt.show()
print('Saved: inference_demo.png')